In [ ]:
# =============== 1. ENVIRONMENT SETUP (LightGBM) ===============
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report, 
                             confusion_matrix, roc_auc_score, roc_curve)
from lightgbm import LGBMClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============== 2. LOAD AND EXPLORE DATA ===============
df = pd.read_csv('data.csv')
print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print('\nDataset Info:')
print(df.info())
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
# =============== 3. PREPROCESSING (MISSING + ENCODING) ===============
def handle_missing_values(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0])
    return df

def encode_categorical_features(df):
    le = LabelEncoder()
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    return df

df = handle_missing_values(df)
df = encode_categorical_features(df)
print('Preprocessing completed. Shape:', df.shape)

In [ ]:
# =============== 4. CREATE TARGET VARIABLE (DEFAULT RISK) ===============
print('Available columns:', df.columns.tolist())

def create_target_variable(df):
    """
    Create a binary default_risk target.
    For this example we approximate risk using arrears-related fields.
    """
    # High risk if there are arrears and NET-OUTSTANDING is positive
    condition = (df['ArrearsCapital'] > 0) | (df['No of Rental in arrears'] > 0)
    df['default_risk'] = np.where(condition, 1, 0)
    return df

df = create_target_variable(df)
print('Target distribution:\n', df['default_risk'].value_counts())
print('\nTarget proportion:\n', df['default_risk'].value_counts(normalize=True))

In [ ]:
# =============== 5. TRAIN / TEST SPLIT ===============
X = df.drop('default_risk', axis=1)
y = df['default_risk']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)

In [ ]:
# =============== 6. TRAIN LIGHTGBM MODEL ===============
lgbm_model = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
lgbm_model.fit(X_train, y_train)
print('LightGBM model trained!')

In [ ]:
# =============== 7. EVALUATION + PD BANDS ===============
y_proba = lgbm_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
print(f'Accuracy: {acc:.4f}')
print(f'ROC AUC: {auc:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Default', 'Default'],
            yticklabels=['Non-Default', 'Default'])
plt.title('Confusion Matrix - LightGBM')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - LightGBM')
plt.legend()
plt.tight_layout()
plt.show()

# PD bands (High >=0.8, Medium 0.2-0.8, Low <0.2)
def pd_to_band(pd_vals):
    bands = []
    for v in pd_vals:
        if v >= 0.8:
            bands.append('High')
        elif v >= 0.2:
            bands.append('Medium')
        else:
            bands.append('Low')
    return pd.Series(bands)

bands = pd_to_band(y_proba)
print('\nPD band counts:')
print(bands.value_counts())

plt.figure(figsize=(5,4))
sns.countplot(x=bands, order=['Low','Medium','High'])
plt.title('PD Risk Bands - LightGBM')
plt.xlabel('Band')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# =============== 8. SAVE MODEL ===============
joblib.dump(lgbm_model, 'lightgbm_model.pkl')
print('Saved LightGBM model to lightgbm_model.pkl')